In [1]:
import numpy as np

np.random.seed(42)

X = np.random.uniform(-2,2, (400,3))

y = (
    np.sin(X[:,0]) +
    0.5 * (X[:,1]**2) -
    0.8 * X[:,2]
)

y = y.reshape(-1,1)

X = X.T
y = y.T

There are 400 samples and 3 input features.
The target function contains sine and quadratic terms, so it is nonlinear.
Because of this, bending will be required.

In [2]:
print("Model A total parameters = 21")
print("Model B total parameters = 73")
print("Model C total parameters = 257")
print("Model D total parameters = 545")

Model A total parameters = 21
Model B total parameters = 73
Model C total parameters = 257
Model D total parameters = 545


Total parameters were calculated manually before coding using:

(number of weights) + (number of biases)

It was observed that parameters increase significantly as depth increases.

In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2


def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha*z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)


def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_derivative(z):
    return sigmoid(z)

All required activation functions and their derivatives were implemented.

Derivatives are necessary because gradients must be computed during backpropagation.

In [4]:
def initialize_parameters(layer_dims):

    parameters = {}

    for l in range(1, len(layer_dims)):

        parameters["W" + str(l)] = np.random.uniform(
            -0.5, 0.5,
            (layer_dims[l], layer_dims[l-1])
        )

        parameters["b" + str(l)] = np.zeros((layer_dims[l], 1))

    return parameters

Weights were initialized randomly between -0.5 and 0.5.
Biases were initialized to zero.

The function was written in general form so it works for any number of layers.

In [11]:
def forward_propagation(X, parameters, activation):

    caches = []
    A = X
    L = len(parameters) // 2

    for l in range(1, L):

        A_prev = A
        W = parameters["W" + str(l)]
        b = parameters["b" + str(l)]

        Z = W @ A_prev + b
        A = activation(Z)

        caches.append((A_prev, W, b, Z))

    # Output layer (linear)
    A_prev = A
    W = parameters["W" + str(L)]
    b = parameters["b" + str(L)]

    Z = W @ A_prev + b
    A = Z

    caches.append((A_prev, W, b, Z))

    return A, caches

Forward pass was implemented using:

Z = W@Aprev + b
A = activation(Z)

Output layer uses linear activation because this is regression.

Intermediate values were stored for backward pass.

In [12]:
def compute_loss(y_hat, y):
    m = y.shape[1]
    loss = (1/m) * np.sum((y_hat - y)**2)
    return loss

Mean Squared Error was used as required.

Squared error ensures larger mistakes contribute more to the loss.

In [17]:
def backward_propagation(y_hat, y, caches, activation_derivative):

    gradients = {}
    L = len(caches)
    m = y.shape[1]

    dA = 2 * (y_hat - y)

    for l in reversed(range(L)):

        A_prev, W, b, Z = caches[l]

        if l == L-1:
            dZ = dA   # output layer (linear)
        else:
            dZ = dA * activation_derivative(Z)

        gradients["dW" + str(l+1)] = (1/m) * (dZ @ A_prev.T)
        gradients["db" + str(l+1)] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        dA = W.T @ dZ

    return gradients

Backward pass was implemented using chain rule.

Gradients were computed layer by layer in reverse order.

Activation derivatives were applied before propagating gradients backward.

In [18]:
def update_parameters(parameters, gradients, learning_rate):

    L = len(parameters) // 2

    for l in range(1, L+1):

        parameters["W" + str(l)] -= learning_rate * gradients["dW" + str(l)]
        parameters["b" + str(l)] -= learning_rate * gradients["db" + str(l)]

    return parameters

Parameters were updated using gradient descent rule.

Each parameter was moved opposite to the gradient direction.

Learning rate used = 0.01

In [19]:
def train_model(layer_dims, activation, activation_derivative, epochs=1000, learning_rate=0.01):

    parameters = initialize_parameters(layer_dims)

    for epoch in range(epochs):

        y_hat, caches = forward_propagation(X, parameters, activation)
        loss = compute_loss(y_hat, y)
        gradients = backward_propagation(y_hat, y, caches, activation_derivative)
        parameters = update_parameters(parameters, gradients, learning_rate)

        if epoch == 199:
            loss_200 = loss

    grad_norm_first = np.linalg.norm(gradients["dW1"])
    grad_norm_last = np.linalg.norm(gradients["dW" + str(len(layer_dims)-1)])

    return loss, loss_200, grad_norm_first, grad_norm_last

Training was done for 1000 epochs.

Loss at epoch 200 was recorded.

Gradient norm of first and last hidden layer was computed using L2 norm.

In [20]:
print("Model A")
print(train_model([3,4,1], relu, relu_derivative))

print("Model B")
print(train_model([3,6,6,1], relu, relu_derivative))

print("Model C")
print(train_model([3,8,8,8,8,1], relu, relu_derivative))

print("Model D with ReLU")
print(train_model([3,8,8,8,8,8,8,8,8,1], relu, relu_derivative))

print("Model D with Sigmoid")
print(train_model([3,8,8,8,8,8,8,8,8,1], sigmoid, sigmoid_derivative))

Model A
(np.float64(0.04066546318397613), np.float64(0.24828954117047297), np.float64(0.033167521327929356), np.float64(0.010790440713697414))
Model B
(np.float64(0.07320978194443808), np.float64(0.6624144525626022), np.float64(0.027564575668806508), np.float64(0.011796298900628503))
Model C
(np.float64(0.04920720901787645), np.float64(0.576386853013729), np.float64(0.017647729015945794), np.float64(0.018669450734757256))
Model D with ReLU
(np.float64(1.704670639152683), np.float64(1.742539437253686), np.float64(0.05988427640532226), np.float64(0.06580123325836972))
Model D with Sigmoid
(np.float64(1.743852386731152), np.float64(1.7438523879467105), np.float64(1.5635982530958202e-06), np.float64(7.844760050118518e-06))


All architectures were trained.

For each model the following were recorded:

Final loss
Loss at epoch 200
Gradient norm of first layer
Gradient norm of last layer

Deep model was run once with ReLU and once with Sigmoid as required.

# Reflections:
## Why does a linear model fail on this dataset?
The dataset contains sine and quadratic terms.
A linear model can only produce a flat surface.
It has a constant slope.
Since the dataset has curvature and oscillation, a linear model cannot fit it properly.

## Did deeper always reduce loss faster?
No.
It was observed that deeper models did not always reduce loss faster.
Sometimes training became slower.

## Did gradients in early layers stay similar to later layers?
No.
In deeper networks, gradients in early layers were usually smaller compared to the last layers.
This shows gradient weakening through layers.

## Was training equally stable for all activations?
No.
Training stability depended on the activation used.

## Which activation behaved more stable in deeper networks?
ReLU behaved more stable in deeper networks.
Sigmoid showed smaller gradients in early layers.

## Which activation behaved more stable in deeper networks?
ReLU behaved more stable in deeper networks.
Sigmoid showed smaller gradients in early layers.